# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas72O5/flyrank-ml-internship_week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

# 1. Detect if we are in Colab
IN_COLAB = "google.colab" in sys.modules

# 2. If in Colab, clone the repository to get the data files
if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter" # Or your own repo URL
    REPO_DIR = "flyrank-ml-internship-starter"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    # Change the current working directory to the repo root
    os.chdir(REPO_DIR)
    print(f"Current directory: {os.getcwd()}")

Current directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule: I am prioritizing pages that are "Champions" (Average Position < 20) and are currently in a "Down" trend. The score is calculated by multiplying the page's Impressions by a binary flag for High Visibility. This ensures we focus on high-traffic assets where a decline is most expensive.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# CODE CELL for Section 1
import pandas as pd
import numpy as np

# Load the starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define our logic components
is_down = df['trend_direction'].str.lower().eq('down')
is_champion = df['avg_position'] < 20
is_stale = df['days_since_last_update'] > 180

print(f"Candidates for Champion Decay: {len(df[is_down & is_champion])}")

Candidates for Champion Decay: 11719


## 2. Build the ranked queue (writes the CSV)


*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Calculate the Score: Only score pages that are actually 'Down'
# Priority = Impressions * (1 if Position < 20 else 0)
df['baseline_score'] = is_down.astype(int) * is_champion.astype(int) * df['impressions_90d']

# Assign Reason Codes and Action Labels
df['reason_code'] = 'CHAMPION_DECAY_RISK'
df.loc[is_stale & is_champion, 'reason_code'] = 'STALE_CHAMPION'
df['action_label'] = 'REFRESH_CONTENT'

# Create the ranked queue (Top 100)
baseline_queue = df.sort_values('baseline_score', ascending=False).head(100)

# Ensure the output directory exists
os.makedirs("../outputs", exist_ok=True)

# Write to CSV
baseline_queue[['content_id', 'baseline_score', 'reason_code', 'action_label']].to_csv("../outputs/baseline_action_score.csv", index=False)

print("Ranked queue written to work/outputs/baseline_action_score.csv")
# Displaying top 5 for verification
display(baseline_queue[['content_id', 'baseline_score', 'reason_code', 'avg_position']])

Ranked queue written to work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,avg_position
6653,content_5fe46e04994d,517715,CHAMPION_DECAY_RISK,4.2
26844,content_8c19996aa890,509252,CHAMPION_DECAY_RISK,2.5
21819,content_4c36c775b818,463103,CHAMPION_DECAY_RISK,2.3
29879,content_1a9e894be2e2,416180,CHAMPION_DECAY_RISK,4.0
13537,content_2c2606c5d176,347399,CHAMPION_DECAY_RISK,4.2
...,...,...,...,...
14955,content_ec66c58d9826,67880,CHAMPION_DECAY_RISK,4.8
6101,content_2340840d66ac,67278,CHAMPION_DECAY_RISK,11.4
26774,content_6b1ceedddf33,66395,CHAMPION_DECAY_RISK,7.3
29775,content_896bf2cc27b7,66359,CHAMPION_DECAY_RISK,4.9


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


### **Top-20 Review: High-Confidence Refresh Candidates**

1. **ID [content_5fe46e04994d]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The keyword volume dropped globally due to seasonality (e.g., a holiday search term ending its peak).
2. **ID [content_8c19996aa890]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: A newer page on the same website is cannibalizing the traffic by ranking for the same term.
3. **ID [content_4c36c775b818]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: Google introduced a new SERP feature (like an AI Overview) that is stealing clicks from the top 3 positions.
4. **ID [content_1a9e894be2e2]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page is currently undergoing a technical migration or backend update that temporarily suppressed traffic.
5. **ID [content_2c2606c5d176]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: A major competitor recently updated their content and our 4.2 ranking is still "settling."
6. **ID [content_cb112fce36be]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: This drop is just standard statistical "wiggle" at position 5.6 which often fluctuates weekly.
7. **ID [content_9532f197bbc8]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The total market interest in this topic has reached a saturation point and demand is naturally dying.
8. **ID [content_008fb02c46cb]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page is failing Core Web Vitals (speed/usability) rather than having a content quality problem.
9. **ID [content_07e0b9af8b1a]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page lost a high-authority external backlink, making a content refresh insufficient to recover rank.
10. **ID [content_cea79ef51519]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: Search intent has shifted (e.g., users now want "Videos" instead of text guides).
11. **ID [content_c8e9d6ab9013]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page is on the verge of falling off Page 1 (9.7) due to a domain-wide authority drop.
12. **ID [content_bf7bff5d0756]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The decline is due to a "Broken Link" or 404 error on a secondary resource the page relies on.
13. **ID [content_9463d30d5826]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The topic is news-sensitive and the "spike" in traffic was a one-time event that has now passed.
14. **ID [content_3d94572c3a35]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page is currently being used for an A/B test and the "Original" is winning but being reported as a decline.
15. **ID [content_01908772c6db]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: A mobile usability error is preventing users from clicking the search result properly.
16. **ID [content_89fcb6f35525]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page is an "Evergreen" reference that is stable in value but currently facing a temporary low-demand month.
17. **ID [content_62ed76850efc]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: A competitor has lowered their product price, causing users to ignore our informational guide for a direct purchase.
18. **ID [content_ab26273a7e7a]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The keyword the page ranks for has become "ambiguous" and Google is testing different types of results.
19. **ID [content_8ba747cf969e]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The page ranks for thousands of tiny keywords and the "average" position is being pulled down by new, irrelevant rankings.
20. **ID [content_f42eb861c6dd]:** Action: Refresh. Code: CHAMPION_DECAY_RISK. Wrong if: The content is physically fine but the page contains too many ads, causing a "Page Layout" penalty from Google.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Top 20 Baseline Candidates for Review:")
display(baseline_queue[['content_id', 'baseline_score', 'avg_position', 'impressions_90d', 'reason_code']].head(20))

Top 20 Baseline Candidates for Review:


,content_id,baseline_score,avg_position,impressions_90d,reason_code
6653,content_5fe46e04994d,517715,4.2,517715,CHAMPION_DECAY_RISK
26844,content_8c19996aa890,509252,2.5,509252,CHAMPION_DECAY_RISK
21819,content_4c36c775b818,463103,2.3,463103,CHAMPION_DECAY_RISK
29879,content_1a9e894be2e2,416180,4.0,416180,CHAMPION_DECAY_RISK
13537,content_2c2606c5d176,347399,4.2,347399,CHAMPION_DECAY_RISK
26531,content_cb112fce36be,309910,5.6,309910,CHAMPION_DECAY_RISK
21565,content_9532f197bbc8,309192,2.0,309192,CHAMPION_DECAY_RISK
27478,content_008fb02c46cb,236803,4.4,236803,CHAMPION_DECAY_RISK
10741,content_07e0b9af8b1a,214816,3.3,214816,CHAMPION_DECAY_RISK
11655,content_cea79ef51519,208798,5.2,208798,CHAMPION_DECAY_RISK


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


**Weak Picks:**
The current rule is heavily biased toward **Impressions**. Because the score is `Flag * Impressions`, a page at Position 15 with 100,000 impressions will always outrank a page at Position 2 with 1,000 impressions. However, the Position 2 page is a much more valuable "Champion" that is closer to the top. The baseline also completely ignores pages marked as "stable"—even if they are about to start declining—creating an opportunity gap.

**Leakage Check:**
- **No Future Windows:** All metrics (`impressions_90d`, `avg_position`) are based on the historical feature window.
- **No Label-Derived Features:** I did not use `trend_pct` in the score calculation.
- **No Product Flags:** I verified that the code does not use `health_score` or any existing FlyRank priority flags, ensuring this is a "from-scratch" baseline.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Leakage Check: Check correlation between baseline_score and the forbidden 'trend_pct'
# A 1.0 correlation would mean we accidentally 'cheated' by using the answer.
leakage_val = df['baseline_score'].corr(df['trend_pct'])
print(f"Correlation between Baseline Score and Trend Pct: {leakage_val:.4f}")

# 2. Show a 'Weak Pick': A high-ranking champion that was ignored because its trend was 'stable'
# These are pages the ML model might correctly flag as 'at risk' that our rule missed.
weak_picks = df[(df['avg_position'] < 5) & (df['baseline_score'] == 0)].head(3)
print("\nWeak Picks (Top-5 Champions ignored by our rule because they aren't 'down' yet):")
display(weak_picks[['content_id', 'avg_position', 'trend_direction', 'baseline_score']])

Correlation between Baseline Score and Trend Pct: -0.0167

Weak Picks (Top-5 Champions ignored by our rule because they aren't 'down' yet):


,content_id,avg_position,trend_direction,baseline_score
10,content_d8ee6cc6d642,2.2,stable,0
11,content_5a3e876cf7f7,0.0,new,0
56,content_dcebfd222b10,4.6,up,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.